In [3]:
from ngsolve import *
from netgen.occ import *
from ngsolve.webgui import Draw

T = 2  # Setze T auf den gewünschten Wert

geo = Box(Pnt(0,0,0), Pnt(1,1,T))  # Box von (0,0,0) bis (1,1,T) # Gitter mit max. Elementgröße 0.2
geo.faces.Max(Z).name = "top"      # Oberseite (z = T)
geo.faces.Min(Z).name = "bottom"   # Unterseite (z = 0)
geo.faces.Min(X).name = "left"     # Seite (x = 0)
geo.faces.Max(X).name = "right"    # Seite (x = 1)
geo.faces.Min(Y).name = "front"    # Seite (y = 0)
geo.faces.Max(Y).name = "back"     # Seite (y = 1)
geo = OCCGeometry(geo) 
mesh = Mesh(geo.GenerateMesh(maxh=0.2)) 
Draw(mesh)


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [4]:
wx = 1 # the wind speed
wy = 1
w =(wx,wy,1) # the wind vector
eps = 1.e-5 # small parameter for the diffusion term

h = specialcf.mesh_size
jac = specialcf.JacobianMatrix(mesh.dim)
invjac = Inv(jac)
M = CF( [[2/sqrt(3), 1/sqrt(3)], [1/sqrt(3), 2/sqrt(3)]] )
expr = invjac.trans * M * invjac

Cinv = 1 # depends on order in general
diffusion = 1

ws = CF(w)
tau =  InnerProduct(ws,expr *ws)
def gradx(u):
    return grad(u)[:-1]

In [9]:
V = VectorH1(mesh, order=1, dirichlet="left|right|front|back|bottom")
Q = H1(mesh, order=1)

X = FESpace([V, Q])

(u, p), (v, q) = X.TnT()

K = BilinearForm(X)
a = eps*gradx(u)*gradx(v)*dx + w*grad(u)*v*dx
asupg = tau*(w*grad(v))*(w*grad(u))*dx
bvp = p*div(v)*dx + tau*grad(p)*w*grad(v)*dx
buq = q*div(u)*dx + tau*grad(q)*w*grad(u)*dx
c = tau *grad(p)*grad(q)*dx
K += a + asupg + bvp + buq + c

IndexError: 